# Installations for Notebook
Required to read Excel file and set schema

In [21]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [22]:
pip install pandera

Note: you may need to restart the kernel to use updated packages.


# Importing Required Modules
Pandera is required for schema validation

In [23]:
import pandas as pd
import pandera.pandas as pa
from pandera.pandas import Column, Check, DataFrameSchema

Read Excel Spreadsheet - Population Estimates at SA2 Level are in sheet "Table 1" 

In [24]:
file_path = "../../external_data/32180DS0003_2001-25.xlsx"

# Load everything as string first - no automatic type inference at all
table1 = pd.read_excel(
    file_path,
    sheet_name="Table 1",
    header=5, # there are 5 rows above that are not related to df
    skipfooter=4, # the last 4 rows are not related to df
    dtype=str
)

table1.head()

,S/T code,S/T name,GCCSA code,GCCSA name,SA4 code,SA4 name,SA3 code,SA3 name,SA2 code,SA2 name,...,no..15,no..16,no..17,no..18,no..19,no..20,no..21,no..22,no..23,no..24
0,1,New South Wales,1RNSW,Rest of NSW,101,Capital Region,10102,Queanbeyan,101021007,Braidwood,...,3950,4041,4145,4218,4282,4332,4368,4389,4480,4494
1,1,New South Wales,1RNSW,Rest of NSW,101,Capital Region,10102,Queanbeyan,101021008,Karabar,...,8531,8530,8516,8500,8535,8548,8527,8470,8432,8409
2,1,New South Wales,1RNSW,Rest of NSW,101,Capital Region,10102,Queanbeyan,101021009,Queanbeyan,...,11230,11362,11460,11468,11460,11375,11378,11391,11344,11251
3,1,New South Wales,1RNSW,Rest of NSW,101,Capital Region,10102,Queanbeyan,101021010,Queanbeyan - East,...,4970,5016,5079,5126,5089,5097,5101,5099,5133,5111
4,1,New South Wales,1RNSW,Rest of NSW,101,Capital Region,10102,Queanbeyan,101021012,Queanbeyan West - Jerrabomberra,...,13150,13090,13022,12955,12821,12748,12790,12861,12872,12735


Drop Irrelevant Rows that do not fit schema (Justification Below)

In [25]:
# Drop the two problem rows outright:
# - Index 2453: Norfolk Island (S/T code = 9, "Other Territories") - not part of 
#   Victoria, and has genuine historical data gaps (".." for 2001-2015) since it 
#   only joined mainland ERP reporting around 2016. 
#  Irrelevant to a Victoria-only project.


# - Index 2454: "Total Australia" - this is a national aggregate/summary row, 
#  not  an actual SA2 area. It was never real geographic data to begin with, so
#  it should never have been validated as an SA2 row in the first place.
table1 = table1.drop(index=[2453, 2454])

print(table1.shape) 

(2453, 35)


Rename Columns no.1, no.2, ..., no.24 to their respective years (2001, 2002, ..., 2025)

In [26]:
old_cols = ['no.'] + [f'no..{i}' for i in range(1, 25)]
new_cols = list(range(2001, 2026))
rename_map = dict(zip(old_cols, new_cols))

table1 = table1.rename(columns=rename_map)

# Define Schema - Dtype, Length and Nullable

In [27]:
schema_dict = {
    # --- Geography codes: stay as strings, fixed-length per ASGS standard ---
    'S/T code':    Column(str, Check.str_length(1, 1), nullable=False),
    'GCCSA code':  Column(str, nullable=False),  # alphanumeric e.g. '1RNSW' - no fixed length check
    'SA4 code':    Column(str, Check.str_length(3, 3), nullable=False),
    'SA3 code':    Column(str, Check.str_length(5, 5), nullable=False),
    'SA2 code':    Column(str, Check.str_length(9, 9), nullable=False),

    # --- Geography names: non-null strings, no length constraint ---
    'S/T name':    Column(str, nullable=False),
    'GCCSA name':  Column(str, nullable=False),
    'SA4 name':    Column(str, nullable=False),
    'SA3 name':    Column(str, nullable=False),
    'SA2 name':    Column(str, nullable=False),
}

In [28]:
# --- Year columns 2001-2025: currently str, coerce to Int64 ---
for year in range(2001, 2026):
    schema_dict[year] = Column(
        'Int64',
        Check.greater_than_or_equal_to(0), # popn >=0
        nullable=False,
        coerce=True
    )

table1_schema = DataFrameSchema(schema_dict, strict=True)

Align Data to Schema Specified Above

In [29]:
try:
    validated = table1_schema.validate(table1, lazy=True)
    print("All rows passed schema validation.")
    table1 = validated  # replace with the coerced version (years now Int64)

# the try block should pass, if not, this helps to investigate rows/cols
# that do not align with schema above
except pa.errors.SchemaErrors as err:
    print(f"{len(err.failure_cases)} schema violations found.\n")
    print(err.failure_cases)

    failing_indices = err.failure_cases['index'].dropna().unique()
    print(f"\nFull rows that failed validation ({len(failing_indices)} rows):")
    print(table1.loc[failing_indices])

All rows passed schema validation.


Filter to Victoria Only

In [30]:
# Filter to Victoria only, using S/T name (readable, self-documenting)
table1_vic = table1[table1['S/T name'] == 'Victoria'].copy()

print(table1_vic.shape)
print(table1_vic['S/T name'].unique())  # should show only 'Victoria'

(522, 35)
['Victoria']


Ensure No Victorian SA2 districts left out - under a slight spelling error

In [31]:
# Rows that got filtered OUT
filtered_out = table1[table1['S/T name'] != 'Victoria']

# Confirm every unique S/T name in the excluded rows is genuinely non-Victoria
print(filtered_out['S/T name'].unique())

# Cross-check: how many rows excluded vs how many rows kept - sum to total
print(f"Kept: {len(table1_vic)}, Excluded: {len(filtered_out)}, "\
      "Total: {len(table1)}")
assert len(table1_vic) + len(filtered_out) == len(table1)

['New South Wales' 'Queensland' 'South Australia' 'Western Australia'
 'Tasmania' 'Northern Territory' 'Australian Capital Territory'
 'Other Territories']
Kept: 522, Excluded: 1931, Total: {len(table1)}


In [32]:
# all of these are outside Victoria - not of our interest so safe to drop
filtered_out[filtered_out['S/T name'] == 'Other Territories']

,S/T code,S/T name,GCCSA code,GCCSA name,SA4 code,SA4 name,SA3 code,SA3 name,SA2 code,SA2 name,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
2450,9,Other Territories,9OTER,Other Territories,901,Other Territories,90101,Christmas Island,901011001,Christmas Island,...,1903,1876,1848,1803,1758,1717,1787,1829,1867,1889
2451,9,Other Territories,9OTER,Other Territories,901,Other Territories,90102,Cocos (Keeling) Islands,901021002,Cocos (Keeling) Islands,...,546,552,553,591,608,603,615,628,622,626
2452,9,Other Territories,9OTER,Other Territories,901,Other Territories,90103,Jervis Bay,901031003,Jervis Bay,...,402,398,386,367,335,309,307,304,304,305


# Remove Duplicate Rows
Since no duplicate SA2 code, all rows must be unique

In [33]:
dupes = table1_vic[table1_vic['SA2 code'].duplicated(keep=False)]
print(f"{len(dupes)} duplicate SA2 rows")
if len(dupes) > 0:
    print(dupes[['SA2 code', 'SA2 name']])

0 duplicate SA2 rows


# Hierarchy Check
Ensuring that all SA2 codes that share the same SA3 code also share the same SA4 and GCCSA codes... 

Otherwise this is a data quality issue

In [34]:
# Only meaningful where the child code can legitimately repeat across rows
checks = [
    ('SA3 code', 'SA4 code'),
    ('SA4 code', 'GCCSA code'),
]

for child, parent in checks:
    n_parents = table1_vic.groupby(child)[parent].nunique()
    bad = n_parents[n_parents > 1]
    print(f"{child} -> {parent}: {len(bad)} inconsistent mappings")

SA3 code -> SA4 code: 0 inconsistent mappings
SA4 code -> GCCSA code: 0 inconsistent mappings


Remove Data Prior to 2012 (only need 2012 onwards)

In [35]:
years_to_drop = list(range(2001, 2012))
table1_vic_recent = table1_vic.drop(columns=years_to_drop)

# Export Data

In [36]:
table1_vic_recent = table1_vic_recent.reset_index(drop=True)
table1_vic_recent.to_csv(
    '../../external_data/victoria_sa2_population_2012_2025_clean.csv', index=False)